In [ ]:
import os
import numpy as np
import pandas as pd


import pickle
from pathlib import Path

In [ ]:
#define global variables
##scratch directory 
##work directory
##work1 ; directory for file from past experiment

data = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/data/2025-11-07_viral_disease_nonseasonal_case_cohort_binning"
results = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/results/2025-11-07_viral_disease_nonseasonal_case_cohort_binning" 
scratch = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/scratch/2025-11-07_viral_disease_nonseasonal_case_cohort_binning"

results1 = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/results/2025-11-05_viral_disease_cohort" 
data1 = "/home/jupyter/workspaces/20251104infectiousdiseasephewasduplicate/data/2025-11-05_viral_disease_cohort" 


#!mkdir {scratch}

In [ ]:
#importing data

#annot csv
def get_cohort_annot():
    #cohort specie and seasonal annotation
    annot_cohort = pd.read_csv(f"{results1}/viral_cohort_person_id_overlap_annotated.csv")

    return annot_cohort
    
#pkl
def get_cohort_dict_pkl(file):

    # …later, in any notebook in the same workspace…
    out_file = Path(f'{data1}/{file}')

    # Reload:
    with open(out_file, 'rb') as f:
        cohort_data_dict = pickle.load(f)

    #print("Reloaded keys:", list(cohort_data_dict.keys()))
    
    return cohort_data_dict



In [ ]:
def get_cohort_updated_race(df):
    
    # Make a copy to avoid modifying the original DataFrame
    wrangled_df = df.copy()

    # --- 1. Merge Race/Ethnicity ---
    
    # Define the helper function for merging
    def merge_race_ethnicity_data(row):
        """Helper function to apply row-wise."""
        if row["ethnicity"] == "Hispanic or Latino":
            return row["ethnicity"]
        else:
            return row["race"]

    # Apply the function to create the 'updated_race' column
    wrangled_df["updated_race"] = wrangled_df.apply(merge_race_ethnicity_data, axis=1)

    # --- 2. Filter Excluded Groups ---
    
    # List of races to exclude
    to_drop = [ 'PMI: Skip',
    'None of these',
    'American Indian or Alaska Native',
    'I prefer not to answer', 'More than one population'
    ]

    # Keep only rows whose updated_race is NOT in the to_drop list
    wrangled_df = wrangled_df[~wrangled_df['updated_race'].isin(to_drop)]


    return wrangled_df

In [ ]:
##1. Bin nonseasonal cohorts

In [ ]:
def filter_dict_by_ns_concept_id(df_dict):
    
    df1 = annot_cohort.copy()
    
    nonseasonal_table =  df1.loc[:, ['condition_concept_id','standard_concept_name','non_seasonal'] ]
    ns = nonseasonal_table.loc[nonseasonal_table['non_seasonal'] == 'Y', :]
    
    # Uses the 'concept_id' column in key_df and in each df
    allowed = pd.to_numeric(ns["condition_concept_id"], errors="coerce").dropna()
    allowed = set(allowed.astype("int64"))

    out = {}
    for name, df in df_dict.items():
        mask = pd.to_numeric(df["condition_concept_id"], errors="coerce").isin(allowed)
        out[name] = df.loc[mask].copy()
    return out


In [ ]:
def get_ns_ancestry_count(nonseasonal_df):
    
    ns_cohort_bin_dict = {}
    
    for key, table in nonseasonal_df.items():
        
        
        race_count = table["updated_race"].value_counts()
        df1 = pd.DataFrame(race_count)
        df1['condition_concept_id'] = key[0] #make a concept_id column with key values
        df1['standard_concept_name'] = key[1]
        df2 = df1.reset_index() #make race a column
        
        ns_cohort_bin_dict[key] = df2
        
        
        
    ns_table = pd.concat(ns_cohort_bin_dict.values(), axis = 0, ignore_index = True)
    new_order = ['condition_concept_id', 'standard_concept_name','updated_race', 'count']
    ns_table = ns_table[new_order]
    
    
    
    
        
    return ns_table

In [ ]:
#takes each concept and assign it to a group based on the # of ancestry groups => 100 

def get_ns_concept_ancestry_bins():
    
    # assume your DataFrame is called df and looks like:
    #   concept_id  concept_name    updated_race  count
    # 0      440029  Viral disease   White         18913
    # …      …       …               …             …

    # 1) Flag which race‐rows meet the ≥100 threshold
    df_filtered = ns_anc_count.copy()

    df_filtered['race_ge_100'] = df_filtered['count'] >= 100

    # 2) Count, per concept, how many races are ≥100
    concept_counts = (
        df_filtered 
          .groupby(['condition_concept_id','standard_concept_name'], as_index=False)
          .agg(n_eligible_races=('race_ge_100','sum'))
    )

    # 3) Bin each concept:
    #    B1: ≥2 races ≥100
    #    B2: exactly 1 race ≥100
    #    A1:  0 races ≥100
    conds = [
        concept_counts['n_eligible_races'] >= 2,
        concept_counts['n_eligible_races'] == 1,
        concept_counts['n_eligible_races'] == 0,
    ]
    choices = ['B1','B2','A1']
    concept_counts['bin'] = np.select(conds, choices, default='A1')

    # 4) (Optional) Merge the bin back onto your original rows
    df_binned = df_filtered.merge(
        concept_counts[['standard_concept_name','bin']],
        on='standard_concept_name',
        how='left'
    )
    
    
    # 5) Inspect
    print(concept_counts['bin'].value_counts())


    for grp in ['B1','B2','A1']:
        members = concept_counts.loc[concept_counts['bin']==grp, ['condition_concept_id','standard_concept_name']]
        print(f"\n=== {grp} ({len(members)} concepts) ===")
        print(members)
        
    return df_binned


In [ ]:
#getting person_Id from filtering

def get_ns_ancestry_bin_person_ids(): 
    master = pd.concat(
        [
            df.assign(
                concept_id=cid_name[0],
                concept_name=cid_name[1]
            )
            for cid_name, df in ns_df.items()
        ],
        ignore_index=True
    )



    # 1) Filter your binned summary to only keep races with ≥100 people
    df_binned_filtered = ns_bin[ns_bin['race_ge_100']]

    # 2) Merge in person_id (only for those high-N rows)
    joined = df_binned_filtered.merge(
        master[
          ['condition_concept_id',
           'standard_concept_name',
           'updated_race',
           'person_id']
        ],
        on=['condition_concept_id',
            'standard_concept_name',
            'updated_race'],
        how='left'
    )

    # 3) Aggregate each (concept, race, bin) into a list of person_ids
    person_lists = (
        joined
        .groupby(
            ['condition_concept_id',
             'standard_concept_name',
             'updated_race',
             'bin'],
            as_index=False
        )['person_id']
        .apply(list)
        .rename(columns={'person_id':'person_ids'})
    )

    # 4) Split into a dict of DataFrames by bin
    dfs_by_bin = {
        b: df_bin.reset_index(drop=True)
        for b, df_bin in person_lists.groupby('bin')
    }

    # ─── (Optional) further nest by concept within each bin ───────────────────────
    nested = {
        b: {
            cid: subdf.drop(columns='bin').reset_index(drop=True)
            for cid, subdf in dfs_by_bin[b].groupby('condition_concept_id')
        }
        for b in dfs_by_bin
    }

    # ─── Usage ─────────────────────────────────────────────────────────────────────
    # DataFrame for bin "A1":
    df_B1 = dfs_by_bin['B1']
    df_B2 = dfs_by_bin['B2']


    return df_binned_filtered, df_B1, df_B2


In [ ]:
# ─── Inputs ────────────────────────────────────────────────────────────────────
# df_binned_filtered: your summary DF already filtered to race_ge_100 == True,
#   with columns ['condition_concept_id','standard_concept_name','updated_race',…]
# ns_df: dict mapping (condition_concept_id, standard_concept_name) → the full person-level DF
#
# ─── Build your triple-keyed dict ──────────────────────────────────────────────

def get_bin_df_for_each_ancestry_and_concept(): 

    filtered_by_triple = {}
    for row in ns_bin_filtered.itertuples(index=False):
        cid, name, race = (
            row.condition_concept_id,
            row.standard_concept_name,
            row.updated_race
        )
        # grab the master DF for that concept
        df_persons = ns_df[(cid, name)]
        # filter it down to just that race
        df_race = df_persons[df_persons['updated_race'] == race].copy()
        # store under the triple key
        filtered_by_triple[(cid, name, race)] = df_race
        
    return filtered_by_triple


In [ ]:
##2. Bin nonseasonal - Vaccinated cohorts

In [ ]:
def get_ns_vax_df_dict_pkl(): 
    # …later, in any notebook in the same workspace…
    out_file = Path(f'{data}/viral_disease_cohort_vaccine_df_dict.pkl')

    # Reload:
    with open(out_file, 'rb') as f:
        cohort_data_dict = pickle.load(f)

    #print("Reloaded keys:", list(cohort_data_dict.keys()))
    
    return cohort_data_dict

In [ ]:
def filter_dict_by_ns_vax_concept_id(df_dict):
    
    df1 = annot_cohort.copy()
    
    nonseasonal_table = df1.loc[:, ['condition_concept_id','standard_concept_name','non_seasonal_vax'] ]
    ns = nonseasonal_table.loc[nonseasonal_table['non_seasonal_vax'] == 'Y', :]
    
    # Uses the 'concept_id' column in key_df and in each df
    allowed = pd.to_numeric(ns["condition_concept_id"], errors="coerce").dropna()
    allowed = set(allowed.astype("int64"))

    out = {}
    
    for name, df in df_dict.items():
        mask = pd.to_numeric(df["condition_concept_id"], errors="coerce").isin(allowed)
        filtered_df = df.loc[mask].copy()

        # Only add the dataframe to the output if it's NOT empty
        if not filtered_df.empty:
            out[name] = filtered_df
            
    return out

In [ ]:
def wrangle_annot_ns_vax_df():
    
    df1 = annot_cohort.copy()
    
    nonseasonal_table = df1.loc[:, ['condition_concept_id','standard_concept_name','non_seasonal_vax',  'specie', 'genus'] ]
    ns_vax = nonseasonal_table.loc[nonseasonal_table['non_seasonal_vax'] == 'Y', :]
    
    
    def merge_specie_data(new_column):

        if new_column["genus"] == "Influenzavirus":
            return new_column["genus"]
        else:
            return new_column["specie"]


    ns_vax_table = ns_vax.copy()
    
    ns_vax_table["updated_specie"] = ns_vax_table.apply(merge_specie_data, axis=1)
    ns_table = ns_vax_table.loc[:, ['condition_concept_id','standard_concept_name', 'updated_specie']]
    ns_table

    return ns_table
    

In [ ]:


merged_dict = {
    key: pd.merge(ns_vax_df[key], vax_df[key], on='person_id', how='inner')
    for key in ns_vax_df
}

In [ ]:
merged_dict

In [ ]:
def add_vax_count_by_ethnicity(new_column):
    
    if new_column["drug_exposure_start_datetime"] is pd.NaT:
        return 'N'
    else:
        return 'Y'


In [ ]:
#Main function calls

#annot_cohort = get_cohort_annot()

#cohort_dict = get_cohort_dict_pkl("viral_disease_cohort_summary_stats_dataset_dict.pkl")

#wrangled_dict = dict()
#for name, df in cohort_dict.items():
   
#    updated = get_cohort_updated_race(df)   
#    wrangled_dict[name] = updated




##ns

#ns_df = filter_dict_by_ns_concept_id(wrangled_dict)
#ns_anc_count = get_ns_ancestry_count(ns_df)
#ns_bin = get_ns_concept_ancestry_bins()
#ns_bin_filtered, df_b1, df_b2 = get_ns_ancestry_bin_person_ids()
#final_ns_bin  = get_bin_df_for_each_ancestry_and_concept()


##ns vax

#vax_df = get_ns_vax_df_dict_pkl()
#ns_vax_df = filter_dict_by_ns_vax_concept_id(wrangled_dict)
#annot_vax_cohort =  wrangle_annot_ns_vax_df()



#for key, table in ns_vax_df.items():

#    table["vaccinated"] = table.apply(add_vax_count_by_ethnicity, axis=1)




#ns_vax_anc_count = get_ns_ancestry_count(ns_vax_df)



In [ ]:
#visulization


#annot_cohort

#cohort_dict

#wrangled_dict


#ns_df
#ns_anc_count
#ns_bin
#concept_counts
#ns_bin_filtered
#df_b1
#df_b2
#final_ns_bin ##save as final group B1 and B2 pkls, A1 is raw df groups


vax_df
#ns_vax_df
#annot_vax_cohort
#annot_vax_cohort["updated_specie"].nunique()
#vax_map 
#concept_vax_id_dict


#ns_vax_anc_count



### save to sratch
annot_vax_cohort.to_csv(f"{scratch}/annot_vax_cohort.csv")
